# Section 3 – Model 1: predict `base_experience` (regression)
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Your ML background: Linear Regression vs Random Forest |
| Estimated time | 60 minutes |
| Produces | `models/regressor.joblib` and `models/regressor_meta.json` |

**Goal:** Train, evaluate and compare a Linear Regression and a Random Forest. Choose the better one with evidence and save it for the API.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## The question
> Given a character's **8 stats**, how much `base_experience` ("power") should it have?

We compare two families of models, exactly like your solubility lab:

| Model | Idea | Strength | Weakness |
|---|---|---|---|
| **Linear Regression** | one straight-line formula, `y = b0 + b1·x1 + ...` | simple, explainable, fast | cannot bend |
| **Random Forest** | hundreds of decision trees that vote | captures curves and interactions | harder to explain, can overfit |

**Rule of this lab:** the simpler model wins unless the forest is *clearly* better.

## Step 1 – Load the clean table and the feature list
(Missing? copy `solution_artifacts/data/` into `data/`.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

clean = pd.read_csv(DATA / "pokemon_clean.csv")
info = json.loads((DATA / "feature_info.json").read_text(encoding="utf-8"))
FEATURES = info["features"]
TARGET = info["targets"]["regression"]
print("features:", FEATURES)
print("target  :", TARGET, "| rows:", len(clean))

## Step 2 – Train / test split
We hide 20% of the data. The model never sees it while learning, so it tells us how the model behaves on **new** characters.

🟦 **Worked sample** (toy data: hero level → hit points):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

rng = np.random.default_rng(0)
level = rng.integers(1, 50, 100)
hp = 50 + 6 * level + rng.normal(0, 15, 100)          # a line plus noise

Xt_train, Xt_test, yt_train, yt_test = train_test_split(level.reshape(-1, 1), hp, test_size=0.2, random_state=42)
toy_model = LinearRegression().fit(Xt_train, yt_train)
pred = toy_model.predict(Xt_test)
print("slope ~", round(toy_model.coef_[0], 2), "(true 6) | R2:", round(r2_score(yt_test, pred), 3),
      "| MAE:", round(mean_absolute_error(yt_test, pred), 1))

In [ ]:
# TODO: X = the feature columns, y = the target column
...  # <- write your code here

# TODO: split into X_train, X_test, y_train, y_test with test_size=0.2 and random_state=42
# HINT: train_test_split(X, y, test_size=0.2, random_state=42)
...  # <- write your code here

check("train is ~80%", abs(len(X_train) / len(clean) - 0.8) < 0.01, len(X_train))
check("target not in the features", TARGET not in X.columns)

## Step 3 – Metrics and a baseline
Three numbers, all on the **test** set:

| Metric | Reads as | Best |
|---|---|---|
| **R²** | share of the variation explained (1 = perfect, 0 = as bad as guessing the mean) | high |
| **MAE** | average error in the target's own units (experience points) | low |
| **RMSE** | like MAE but punishes big mistakes more | low |

A **baseline** (always predict the mean) tells us what "no intelligence" scores. Any model must beat it.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def evaluate(model, X_eval, y_eval):
    """Return a dict with r2, mae and rmse of the model on the given data."""
    # TODO: predict and compute the three metrics (rmse = square root of the mean squared error)
    ...  # <- write your code here
    return {"r2": round(float(r2), 4), "mae": round(float(mae), 2), "rmse": round(rmse, 2)}


baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
results = {"baseline (mean)": evaluate(baseline, X_test, y_test)}
print(results)
check("baseline R2 is about 0", abs(results["baseline (mean)"]["r2"]) < 0.05)

## Step 4 – Model A: Linear Regression
Linear models are sensitive to the **scale** of the features (weight is in the thousands, speed in the tens),
so we chain a `StandardScaler` and the model in a **Pipeline**. The pipeline is one object that we can save and call later with raw numbers.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TODO: pipeline = StandardScaler followed by LinearRegression; fit it on the TRAIN data only
# HINT: make_pipeline(StandardScaler(), LinearRegression())
...  # <- write your code here

results["Linear Regression"] = evaluate(lin, X_test, y_test)
print(results["Linear Regression"])

coef = pd.Series(lin[-1].coef_, index=FEATURES).sort_values()
print("\nCoefficients (per 1 standard deviation):")
print(coef.round(2))
check("linear model beats the baseline", results["Linear Regression"]["r2"] > 0.3, results["Linear Regression"]["r2"])

**Read the coefficients:** a positive number means "more of this feature → more experience", holding the others fixed.
Which stat matters most? Does it make sense for a game?

## Step 5 – Model B: Random Forest
Trees do not need scaling. `n_estimators` is the number of trees; `random_state` makes the result reproducible.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: RandomForestRegressor with 300 trees, random_state=42 and n_jobs=-1; fit on the TRAIN data only
...  # <- write your code here

results["Random Forest"] = evaluate(rf, X_test, y_test)
print(results["Random Forest"])

importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
importance.plot.barh(color="#0066cc", title="Random Forest feature importance", figsize=(6, 3.5))
plt.show()
check("forest beats the baseline", results["Random Forest"]["r2"] > 0.3, results["Random Forest"]["r2"])

## Step 6 – Do not trust one split: cross-validation
One lucky or unlucky split can mislead. **5-fold cross-validation** trains and tests 5 times on different parts and reports the average.

🟦 **Worked sample** with the toy model:

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LinearRegression(), level.reshape(-1, 1), hp, cv=5, scoring="r2")
print("fold scores:", scores.round(3), "| mean:", scores.mean().round(3), "| std:", scores.std().round(3))

In [ ]:
# TODO: 5-fold CV of R2 for both pipelines on the WHOLE X, y. Store (mean, std) in cv_lin and cv_rf
# HINT: cross_val_score(model, X, y, cv=5, scoring="r2")
...  # <- write your code here

cv_lin = (cv_lin_scores.mean(), cv_lin_scores.std())
cv_rf = (cv_rf_scores.mean(), cv_rf_scores.std())
print(f"Linear Regression CV R2: {cv_lin[0]:.3f} ± {cv_lin[1]:.3f}")
print(f"Random Forest     CV R2: {cv_rf[0]:.3f} ± {cv_rf[1]:.3f}")
results["Linear Regression"]["cv_r2"] = round(float(cv_lin[0]), 4)
results["Random Forest"]["cv_r2"] = round(float(cv_rf[0]), 4)

## Step 7 – Compare (table and picture)

In [ ]:
table = pd.DataFrame(results).T
print(table)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, [("Linear Regression", lin), ("Random Forest", rf)]):
    ax.scatter(y_test, model.predict(X_test), s=12, alpha=0.6, color="#0066cc")
    lims = [y.min(), y.max()]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set_title(f"{name}  (R2={results[name]['r2']})")
    ax.set_xlabel("actual base_experience")
axes[0].set_ylabel("predicted")
plt.tight_layout()
plt.show()

**Read the plot:** points on the red dashed line are perfect predictions. Which model hugs it more? Where are the largest errors (very strong characters?).

## Step 8 (optional but recommended) – Tune the forest a little
`GridSearchCV` tries combinations of settings with cross-validation. Keep the grid **small**.

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: grid over n_estimators [200, 400], max_depth [None, 10] and min_samples_leaf [1, 3]; cv=3, scoring="r2"
# HINT: GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid, cv=3, scoring="r2").fit(X_train, y_train)
...  # <- write your code here

print("best settings:", grid.best_params_)
results["Random Forest (tuned)"] = evaluate(grid.best_estimator_, X_test, y_test)
print(results["Random Forest (tuned)"])

## Step 9 – Choose the winner and save it
Decision rule: pick the model with the **lowest test MAE**. If the forest wins by less than **5%**, prefer the linear model
(simpler = easier to explain and to maintain). Write one sentence explaining your choice.

`joblib.dump` saves the whole pipeline to a file; the API will load it once at startup (Section 5).

In [ ]:
import joblib
import sklearn
from datetime import datetime

candidates = {"Linear Regression": lin, "Random Forest": rf}
if "Random Forest (tuned)" in results:
    candidates["Random Forest (tuned)"] = grid.best_estimator_

# TODO: best_name = candidate with the lowest test MAE; if a forest wins by less than 5% over Linear Regression keep Linear Regression
# HINT: results[name]["mae"]
...  # <- write your code here

best_model = candidates[best_name]
joblib.dump(best_model, MODELS / "regressor.joblib")

meta = {
    "name": best_name, "target": TARGET, "features": FEATURES,
    "metrics": results[best_name], "all_results": results,
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "trained_at": datetime.now().isoformat(timespec="seconds"), "sklearn_version": sklearn.__version__,
}
(MODELS / "regressor_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved:", best_name, results[best_name])

In [ ]:
# Re-load from disk and predict one new character (this is what the API will do)
import joblib

loaded = joblib.load(MODELS / "regressor.joblib")
mewtwo = pd.DataFrame([{"hp": 106, "attack": 110, "defense": 90, "special_attack": 154,
                        "special_defense": 90, "speed": 130, "height": 20, "weight": 1220}])
print("Predicted base_experience for a mewtwo-like character:", round(float(loaded.predict(mewtwo[FEATURES])[0]), 1))
check("model file saved", (MODELS / "regressor.joblib").exists())
check("meta file saved", (MODELS / "regressor_meta.json").exists())
check("the saved model beats the baseline", meta["metrics"]["r2"] > 0.3, meta["metrics"]["r2"])

## Questions
1. Which model won and by how much? Is the difference worth a more complex model?
2. Look at the scatter plot: for which kind of characters is the error largest?
3. Why do we fit the scaler on the **training** data only? What would happen if it saw the test data?
4. The card will print "Predicted power ± MAE". How would you explain that error to a player?

### Extra challenge
Add `total_stats` as a feature (Section 2 challenge) and re-run Steps 4–6. Which model benefits more, and why?

**Next:** Section 4 – the classification models (is it legendary?).